# 🧬 Notebook 01 — BLAST Alignment and Species Parsing

**What this notebook covers**  
This notebook walks through the core of the eDNA pipeline: how a raw DNA sequence becomes a species identification. You will learn what BLAST is doing under the hood, how the pipeline parses the results, and how species names are extracted and cleaned from messy BLAST output. By the end you will be able to run a complete parse on synthetic data and read the resulting table.

**Who it is for**  
ANYONE TRYING TO LEARN. No need to be a software engineer to follow this. Every code cell is commented line by line. You can run all cells top to bottom with no internet connection and no database installed.

---

## Section 1 — The Biological Problem

### What is eDNA?

Environmental DNA (eDNA) is genetic material shed by organisms into their surroundings — water, soil, air. A fish swimming through a pond leaves behind cells from its skin, faeces, and mucus. Those cells disintegrate, but the DNA fragments persist in the water column for hours to weeks. By filtering a water sample and extracting the DNA, you can detect which species were present without ever seeing or catching them.

### Why does sequence alignment matter?

After extraction and PCR amplification, a sequencing machine gives you a string of letters — the nucleotide sequence. On its own that string is meaningless. To identify the species, you compare (align) the unknown sequence against a reference database of millions of known sequences. The species whose reference sequence most closely matches your unknown sequence is your best identification.

### Real-world context — UK pond survey

Imagine you are surveying a lowland pond in the UK for the presence of great crested newt (*Triturus cristatus*), a protected species under the Wildlife and Countryside Act 1981. You filter 3 litres of water, extract the DNA, amplify the 12S ribosomal RNA gene, and sequence the product. The pipeline takes that sequence, runs BLAST against the NCBI nucleotide (nt) database, and tells you whether a great crested newt has been using that pond.

This notebook uses freshwater pond organisms — green algae, cyanobacteria, water fleas, and midges — as the worked examples, because they are the exact organisms that appear in the `mock_run.py` demonstration.

---

**Key term:** A **barcode gene** is a short, standardised region of DNA that is:
- Conserved enough that PCR primers work across many species in the same group
- Variable enough between species to tell them apart

| Barcode gene | Organism group | Length (bp) |
|---|---|---|
| 16S rRNA | Bacteria (cyanobacteria) | ~400 |
| 18S rRNA | Eukaryotes (algae, invertebrates) | ~400–600 |
| COI (cytochrome oxidase I) | Animals | ~650 |
| trnL intron | Plants (macrophytes) | ~300 |

---

## Section 2 — How BLAST Works

BLAST stands for **Basic Local Alignment Search Tool**. It is not trying to find the perfect global alignment between two sequences — that would be too slow for millions of database entries. Instead it uses a fast heuristic in three stages.

### Step 1 — Seeding

BLAST breaks your query sequence into short overlapping fragments called **words** (typically 11 nucleotides for blastn). It looks up every word in a pre-built index of the database and instantly finds all positions in all database sequences that share an exact or near-exact match. This is the fast step — it narrows 10 billion base pairs down to thousands of candidate positions.

### Step 2 — Extension (ungapped, then gapped)

Each seed match is extended left and right to form a **High-Scoring Pair (HSP)**. BLAST first extends without allowing gaps, then re-runs with gaps (insertions and deletions) to recover the best possible local alignment. Extension stops when the score falls too far below the running maximum.

### Step 3 — Scoring and filtering

BLAST scores each HSP using a substitution matrix (for blastn: +1 for a match, -3 for a mismatch) and penalises gaps. HSPs below a minimum score are discarded. The remaining alignments are ranked and reported.

---

### The 4 key metrics the pipeline uses

| Metric | What it measures | Good value | Pipeline threshold |
|---|---|---|---|
| **Identity %** | Percentage of aligned bases that match exactly | ≥97% for species ID | ≥97% = high, ≥90% = medium, <90% = low |
| **E-value** | Expected number of equally good hits by chance in the database | < 1×10⁻¹⁰ | Default filter: 1×10⁻¹⁰ |
| **Bit score** | Normalised alignment score (independent of database size) | Higher = better | Used to rank HSPs |
| **Query coverage %** | Fraction of your query that is covered by the alignment | ≥80% | Capped at 100% |

### Worked example

You submit a 480 bp 18S rRNA sequence. BLAST returns:

| Rank | Hit species | Identity % | E-value | Bit score | Query cov % | Interpretation |
|---|---|---|---|---|---|---|
| 1 | *Chlorella vulgaris* | 98.5 | 1×10⁻²¹⁰ | 850 | 98.3 | Strong species-level match |
| 2 | *Chlorella sorokiniana* | 97.1 | 1×10⁻¹⁹⁵ | 805 | 97.5 | Close relative, same genus |
| 3 | *Chlorella variabilis* | 96.8 | 1×10⁻¹⁹⁰ | 790 | 96.9 | Also close, slightly lower |

The top hit has 98.5% identity — above the 97% threshold — so the pipeline assigns **species-level confidence** and names this sequence *Chlorella vulgaris*.

---

## Section 3 — Setup

Run this cell first. It imports the standard libraries and the project's own modules. The `sys.path.insert` line tells Python to look for modules in the project root directory, so `from src.parser import ...` works regardless of where Jupyter was launched from.

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

import pandas as pd

# Add the project root to the Python path so we can import from src/
# __file__ inside a notebook is not always defined, so we use Path.cwd()
# Assumes the notebook is opened from inside the notebooks/ directory,
# or that Jupyter was launched from the project root.
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the two pipeline modules this notebook focuses on
from src.parser import _extract_species, parse_all_results
from src.summarise import build_hit_table, calculate_diversity, resolve_taxonomy

print(f"Project root: {PROJECT_ROOT}")
print("Imports successful.")

---

## Section 4 — Running your first pipeline

The pipeline normally receives XML files produced by BLAST. Here we create three synthetic XML files by hand — one per pond sequence — using the same XML structure that real BLAST produces. This lets you run the complete parse without any database or network access.

We use Python's `tempfile.TemporaryDirectory` so the files are automatically deleted after the cell runs.

### The synthetic XML approach

The helper functions below (`_hsp`, `_hit`, `_blast_xml`) are copied from `mock_run.py`. They build a valid BLAST XML string from simple parameters: species name, identity percentage, alignment length, e-value, and bit score.

In [ ]:
# ── Helper functions to build synthetic BLAST XML ───────────────────────────
# These produce exactly the XML format that Biopython's NCBIXML parser expects.

def _hsp(identity_count, align_len, query_len, evalue, bits):
    """Build the <Hsp> XML block for one High-Scoring Pair."""
    return textwrap.dedent(f"""\
            <Hsp>
              <Hsp_num>1</Hsp_num>
              <Hsp_bit-score>{bits}</Hsp_bit-score>
              <Hsp_score>{int(bits)}</Hsp_score>
              <Hsp_evalue>{evalue}</Hsp_evalue>
              <Hsp_query-from>1</Hsp_query-from>
              <Hsp_query-to>{query_len}</Hsp_query-to>
              <Hsp_hit-from>1</Hsp_hit-from>
              <Hsp_hit-to>{align_len}</Hsp_hit-to>
              <Hsp_query-frame>1</Hsp_query-frame>
              <Hsp_hit-frame>1</Hsp_hit-frame>
              <Hsp_identity>{identity_count}</Hsp_identity>
              <Hsp_positive>{identity_count}</Hsp_positive>
              <Hsp_gaps>0</Hsp_gaps>
              <Hsp_align-len>{align_len}</Hsp_align-len>
              <Hsp_qseq>ACGT</Hsp_qseq>
              <Hsp_hseq>ACGT</Hsp_hseq>
              <Hsp_midline>||||</Hsp_midline>
            </Hsp>""")


def _hit(num, accession, title, identity_pct, align_len, query_len, evalue, bits):
    """Build the <Hit> XML block for one database match."""
    identity_count = int(align_len * identity_pct / 100)  # convert % to raw count
    return textwrap.dedent(f"""\
        <Hit>
          <Hit_num>{num}</Hit_num>
          <Hit_id>gi|{num * 1000}|gb|{accession}.1|</Hit_id>
          <Hit_def>{title}</Hit_def>
          <Hit_accession>{accession}</Hit_accession>
          <Hit_len>{align_len + 8}</Hit_len>
          <Hit_hsps>
{_hsp(identity_count, align_len, query_len, evalue, bits)}
          </Hit_hsps>
        </Hit>""")


def _blast_xml(query_id, query_def, query_len, hits):
    """
    Build a complete BLAST XML document for one query.
    hits: list of (accession, title, identity_pct, align_len, evalue, bits)
    """
    hit_blocks = "\n".join(
        _hit(i + 1, acc, title, ident, alen, query_len, ev, bits)
        for i, (acc, title, ident, alen, ev, bits) in enumerate(hits)
    )
    return textwrap.dedent(f"""\
<?xml version="1.0"?>
<BlastOutput>
  <BlastOutput_program>blastn</BlastOutput_program>
  <BlastOutput_version>BLASTN 2.14.0+</BlastOutput_version>
  <BlastOutput_reference>Reference</BlastOutput_reference>
  <BlastOutput_db>nt</BlastOutput_db>
  <BlastOutput_query-ID>Query_1</BlastOutput_query-ID>
  <BlastOutput_query-def>{query_id} {query_def}</BlastOutput_query-def>
  <BlastOutput_query-len>{query_len}</BlastOutput_query-len>
  <BlastOutput_param>
    <Parameters>
      <Parameters_matrix></Parameters_matrix>
      <Parameters_expect>10</Parameters_expect>
      <Parameters_sc-match>1</Parameters_sc-match>
      <Parameters_sc-mismatch>-3</Parameters_sc-mismatch>
      <Parameters_gap-open>5</Parameters_gap-open>
      <Parameters_gap-extend>2</Parameters_gap-extend>
      <Parameters_filter>L;m;</Parameters_filter>
    </Parameters>
  </BlastOutput_param>
  <BlastOutput_iterations>
    <Iteration>
      <Iteration_iter-num>1</Iteration_iter-num>
      <Iteration_query-ID>Query_1</Iteration_query-ID>
      <Iteration_query-def>{query_id} {query_def}</Iteration_query-def>
      <Iteration_query-len>{query_len}</Iteration_query-len>
      <Iteration_hits>
{hit_blocks}
      </Iteration_hits>
      <Iteration_stat>
        <Statistics>
          <Statistics_db-num>1000</Statistics_db-num>
          <Statistics_db-len>1000000</Statistics_db-len>
          <Statistics_hsp-len>0</Statistics_hsp-len>
          <Statistics_eff-space>0</Statistics_eff-space>
          <Statistics_kappa>0.041</Statistics_kappa>
          <Statistics_lambda>0.267</Statistics_lambda>
          <Statistics_entropy>0.14</Statistics_entropy>
        </Statistics>
      </Iteration_stat>
    </Iteration>
  </BlastOutput_iterations>
</BlastOutput>
""")

print("Helper functions defined.")

In [ ]:
# ── Define 3 simple synthetic pond sequences ────────────────────────────────
# Each entry: (query_id, query_def, query_length_bp, list_of_hits)
# Each hit:   (accession, blast_title, identity_pct, align_len, evalue, bits)

SIMPLE_SAMPLES = [
    (
        "SAMPLE_A",          # our query ID
        "18S_green_alga",    # short description
        480,                 # query length in base pairs
        [
            # High-confidence hit: 98.5% identity → species assignment
            ("KX580994",
             "Chlorella vulgaris strain CCAP 211 18S ribosomal RNA gene partial",
             98.5, 472, "1e-210", 850.0),
        ],
    ),
    (
        "SAMPLE_B",
        "16S_cyanobacterium",
        410,
        [
            # High-confidence hit: toxin-producing cyanobacterium
            ("AB001339",
             "Microcystis aeruginosa strain NIES-843 16S ribosomal RNA gene",
             99.2, 406, "1e-195", 800.0),
        ],
    ),
    (
        "SAMPLE_C",
        "trnL_macrophyte",
        300,
        [
            # High-confidence hit: common wetland reed
            ("AJ232936",
             "Phragmites australis voucher PA44 trnL intron partial sequence",
             97.8, 293, "1e-130", 530.0),
        ],
    ),
]

# ── Write XML files to a temporary directory, then parse them ───────────────
with tempfile.TemporaryDirectory() as tmpdir:
    xml_dir = Path(tmpdir) / "xml"
    xml_dir.mkdir()  # create the xml/ subdirectory

    # Write one XML file per sample
    for query_id, query_def, query_len, hits in SIMPLE_SAMPLES:
        xml_path = xml_dir / f"{query_id}.xml"   # e.g. SAMPLE_A.xml
        xml_path.write_text(
            _blast_xml(query_id, query_def, query_len, hits),
            encoding="utf-8",
        )
        print(f"Wrote {xml_path.name}")

    # parse_all_results reads every *.xml in the directory and returns
    # a list of BlastHit dataclass objects
    all_hits = parse_all_results(xml_dir, max_hits=5, evalue_threshold=1e-10)
    print(f"\nParsed {len(all_hits)} BlastHit record(s)")

    # build_hit_table converts the list to a tidy pandas DataFrame
    # It also runs LCA taxonomy resolution internally (see Section 6)
    df = build_hit_table(all_hits)

# Store df outside the temp context so we can use it in later cells
# (the XML files are deleted when the 'with' block exits, but df lives in memory)
SIMPLE_DF = df.copy()
print(f"DataFrame shape: {SIMPLE_DF.shape} (rows × columns)")

In [ ]:
# Display the DataFrame — each row is one BLAST hit
# The most important columns:
#   query_id         : which sequence this hit came from
#   species          : raw name extracted from BLAST title
#   identity_pct     : % of aligned bases that matched
#   confidence_flag  : high / medium / low based on identity thresholds
#   resolved_species : final taxonomic assignment after LCA logic

SIMPLE_DF[[
    "query_id", "species", "identity_pct",
    "evalue", "bit_score", "confidence_flag", "resolved_species"
]].head(10)

---

## Section 5 — Species name parsing deep-dive

BLAST hit titles are messy. The database entry for *Chlorella vulgaris* might be titled:

```
gb|KX580994.1| Chlorella vulgaris strain CCAP 211 18S ribosomal RNA gene partial sequence
```

The pipeline needs to extract just `Chlorella vulgaris` from that string. The function `_extract_species()` in `src/parser.py` handles this using a set of rules:

1. Strip accession prefixes (`gb|`, `emb|`, etc.)
2. Find the first token that starts with an uppercase letter and is alphabetic — that is the **genus**
3. Look at the next token — if it starts with a lowercase letter and is alphabetic, that is the **species epithet**
4. Skip taxonomic qualifiers between genus and epithet (`cf.`, `aff.`, `var.`, `subsp.`, etc.)
5. Stop if a descriptor word is encountered (`clone`, `isolate`, `partial`, etc.)
6. Handle uncultured sequences by rescuing the genus name
7. Fall back to bracket notation `[Organism name]` if nothing else works

The cell below tests `_extract_species` on six different title formats and shows what comes out.

In [ ]:
# Six representative BLAST title formats and what _extract_species does with them

test_cases = [
    # (title_string, expected_output, explanation)
    (
        "gb|KX580994.1| Chlorella vulgaris strain CCAP 211 18S ribosomal RNA partial",
        "Standard format: accession prefix stripped, genus+epithet found directly"
    ),
    (
        "uncultured Chlamydomonas sp. clone ENV45 18S ribosomal RNA partial",
        "Uncultured rescue: 'uncultured' is in _NON_SPECIFIC_TERMS so it is skipped; "
        "Chlamydomonas is found as the genus; 'sp' is a skip qualifier so epithet search "
        "continues; 'clone' is a stop word, so we return genus only → Chlamydomonas sp."
    ),
    (
        "Alternaria cf. alternata strain CBS 147.52 ITS region partial",
        "'cf.' qualifier: means 'compare with' — taxonomist is uncertain. "
        "_SKIP_QUALIFIERS contains 'cf', so the parser skips it and finds "
        "'alternata' as the epithet → Alternaria alternata"
    ),
    (
        "Populus x canadensis clone PC7 trnL intron partial sequence",
        "Hybrid notation: '×' (or 'x') separates parent names in hybrids. "
        "The × is replaced with a space; 'x' is in _SKIP_QUALIFIERS, so it is "
        "skipped, and 'canadensis' is found → Populus canadensis"
    ),
    (
        "gb|AY485484.1| Fragilaria capucina strain FC1 18S partial >gb|AY485485.1| Synedra ulna isolate",
        "Multi-hit title: BLAST can join multiple database entries with '>'. "
        "Only the text before the first '>' is used → Fragilaria capucina"
    ),
    (
        "Freshwater metagenome, whole genome shotgun sequence [metagenome]",
        "Metagenome: 'metagenome' is in _NON_SPECIFIC_BRACKETS, so the "
        "bracket fallback also returns nothing → unclassified"
    ),
]

# Run each title through _extract_species and print the result with explanation
print(f"{'INPUT TITLE':<70}  {'EXTRACTED NAME':<30}")
print("-" * 104)

for title, explanation in test_cases:
    result = _extract_species(title)   # call the real pipeline function
    # Truncate title for display
    display_title = (title[:67] + "...") if len(title) > 70 else title
    print(f"{display_title:<70}  {result:<30}")
    print(f"  → Why: {explanation}")
    print()

---

## Section 6 — LCA taxonomy resolution

### The problem with a single BLAST hit

If you have a 93% identity match to *Daphnia pulex*, should you report *Daphnia pulex*? Not necessarily. At 93% identity, several other *Daphnia* species could produce an equally good alignment — the sequence is just not different enough to pin it to a single species. Trusting the top hit blindly would introduce false species identifications.

### The LCA (Lowest Common Ancestor) approach

Instead of taking the top hit, the pipeline examines **all hits within a small identity window** of the top score (default: 2%). If all those hits agree on the same genus but point to different species, the pipeline reports the **genus** rather than a specific species. This is the LCA — the most specific name that is supported by all the evidence.

The logic in `resolve_taxonomy()` (in `src/summarise.py`) has three tiers:
- **≥97% identity** → trust the species name from the top hit directly
- **90–96% identity** → check genus consensus within the identity window → report genus
- **<90% identity** → check genus consensus across all hits → report genus, or `unclassified` if genera disagree

The cell below creates a small DataFrame with three hits for the same query, all at 93/92/91% identity pointing to different *Daphnia* species.

In [ ]:
# Build a small DataFrame manually to demonstrate LCA resolution
# This represents one eDNA query (POND_004) that got three different Daphnia hits

daphnia_data = {
    "query_id":           ["POND_004", "POND_004", "POND_004"],
    "query_length":       [390, 390, 390],
    "hit_rank":           [1, 2, 3],   # rank 1 = best hit
    "accession":          ["AJ130819", "AJ130820", "AJ130821"],
    "species":            ["Daphnia pulex", "Daphnia magna", "Daphnia longispina"],
    "description":        [
        "Daphnia pulex isolate DP4 18S ribosomal RNA partial sequence",
        "Daphnia magna isolate DM12 18S ribosomal RNA partial",
        "Daphnia longispina clone DL7 18S ribosomal RNA partial sequence",
    ],
    "identity_pct":       [93.4, 92.8, 91.9],   # all medium-confidence (90–96%)
    "evalue":             [1e-140, 1e-135, 1e-128],
    "bit_score":          [580.0, 560.0, 535.0],
    "alignment_length":   [364, 361, 358],
    "query_coverage_pct": [93.3, 92.6, 91.8],
    "confidence_flag":    ["medium", "medium", "medium"],
}

# Create the DataFrame
daphnia_df = pd.DataFrame(daphnia_data)

print("Input DataFrame (before LCA resolution):")
print(daphnia_df[["query_id", "hit_rank", "species", "identity_pct", "confidence_flag"]])

In [ ]:
# Run resolve_taxonomy() — this is the same function the pipeline calls internally
resolved_df = resolve_taxonomy(daphnia_df)

print("After LCA resolution:")
print(resolved_df[["query_id", "hit_rank", "species", "identity_pct", "resolved_species"]])
print()

# Show just the resolved name for the top hit
top_hit = resolved_df[resolved_df["hit_rank"] == 1].iloc[0]
print(f"Raw BLAST top hit  : {top_hit['species']}")
print(f"Resolved species   : {top_hit['resolved_species']}")
print()
print("What happened:")
print("  Top identity was 93.4% — this is in the 'medium confidence' band (90–96%).")
print("  resolve_taxonomy looked at all hits within 2% of the top score (91.4–93.4%).")
print("  All three hits in that window share the same genus: Daphnia.")
print("  Genus consensus achieved → resolved to 'Daphnia sp.' instead of 'Daphnia pulex'.")
print("  This prevents a false species-level ID when the sequence could match multiple species.")

---

## Section 7 — Diversity metrics

Once every sequence has been assigned a species (or genus), the pipeline computes **alpha diversity** — a measure of how many different species are in the sample and how evenly distributed they are.

### Shannon–Wiener Index (H')

H' = −Σ p_i · ln(p_i)

Where p_i is the proportion of sequences belonging to species i. A community with 10 species all equally common has a higher H' than one where 9 species are rare and 1 dominates.

### Pielou's Evenness (J')

J' = H' / ln(S)

Where S is species richness. J' ranges from 0 (one species dominates completely) to 1 (all species are equally common). It tells you not just *how many* species there are, but *how evenly distributed* the reads are.

In [ ]:
# Build a hit table with 4 species across 8 queries
# This represents a small pond sample with uneven species distribution

diversity_data = {
    "query_id":           [f"Q{i:02d}" for i in range(1, 9)],
    "query_length":       [480] * 8,
    "hit_rank":           [1] * 8,     # all rank-1 hits
    "accession":          ["ACC001", "ACC002", "ACC003", "ACC004",
                           "ACC005", "ACC006", "ACC007", "ACC008"],
    # Species distribution: Chlorella 4x, Microcystis 2x, Phragmites 1x, Daphnia 1x
    # This is intentionally uneven to show H' < ln(4)
    "resolved_species":   [
        "Chlorella vulgaris", "Chlorella vulgaris",
        "Chlorella vulgaris", "Chlorella vulgaris",
        "Microcystis aeruginosa", "Microcystis aeruginosa",
        "Phragmites australis",
        "Daphnia sp.",
    ],
    "species":            [
        "Chlorella vulgaris", "Chlorella vulgaris",
        "Chlorella vulgaris", "Chlorella vulgaris",
        "Microcystis aeruginosa", "Microcystis aeruginosa",
        "Phragmites australis",
        "Daphnia sp.",
    ],
    "identity_pct":       [98.5, 98.1, 97.9, 97.6, 99.2, 99.0, 97.8, 93.4],
    "evalue":             [1e-210, 1e-205, 1e-200, 1e-195,
                           1e-195, 1e-193, 1e-130, 1e-140],
    "bit_score":          [850.0, 835.0, 820.0, 810.0, 800.0, 795.0, 530.0, 580.0],
    "alignment_length":   [472, 470, 469, 468, 406, 411, 293, 364],
    "query_coverage_pct": [98.3, 97.9, 97.7, 97.5, 99.0, 100.0, 97.7, 93.3],
    "confidence_flag":    ["high", "high", "high", "high",
                           "high", "high", "high", "medium"],
}

div_df = pd.DataFrame(diversity_data)

# calculate_diversity expects the resolved_species column and hit_rank == 1
diversity = calculate_diversity(div_df)

print("Diversity metrics:")
for key, value in diversity.items():
    print(f"  {key:<30}: {value}")

In [ ]:
import math

# ── Plain-English interpretation ─────────────────────────────────────────────
S = diversity["species_richness"]
H = diversity["shannon_index"]
J = diversity["pielou_evenness"]

# The theoretical maximum H' for S species is ln(S)
H_max = round(math.log(S), 4) if S > 1 else 1.0

print(f"Species richness (S) : {S}")
print(f"  → {S} distinct taxa detected in this sample.")
print()
print(f"Shannon H'           : {H}  (maximum possible for {S} species = {H_max})")
print(f"  → H' would equal {H_max:.4f} if all 4 species were equally common.")
print(f"  → Our H' of {H} is lower because Chlorella dominates (4 out of 8 reads).")
print()
print(f"Pielou J'            : {J}  (range 0–1, where 1 = perfectly even)")
print(f"  → J' = {J} means the community is moderately uneven.")
print(f"  → Chlorella makes up 50% of reads; a perfectly even community would be 25% each.")
print()
print("Ecological interpretation:")
print("  This pond sample is algae-dominated (Chlorella + Microcystis = 75% of eDNA reads).")
print("  The presence of Microcystis aeruginosa (a toxin-producing cyanobacterium)")
print("  alongside Daphnia (a filter-feeder) and Phragmites (emergent macrophyte)")
print("  is consistent with a eutrophic lowland pond in the UK.")

---

## Section 8 — Exercises

These exercises are designed to deepen your understanding. Try them after reading through the notebook once.

### Exercise 1 — Identity thresholds

In Section 4, all three samples produced `high` confidence hits. What would happen if you changed the identity on `SAMPLE_A` from 98.5% to **91.5%**?

1. Modify `SIMPLE_SAMPLES` in Section 4 and change the identity percentage for SAMPLE_A's hit to 91.5.
2. Re-run the parse.
3. Check the `confidence_flag` and `resolved_species` columns in the output DataFrame.
4. Why does `resolved_species` become `Chlorella sp.` instead of `Chlorella vulgaris`? Which rule in `resolve_taxonomy()` triggers?

### Exercise 2 — Add a new species

The UK's most iconic eDNA target is the **great crested newt** (*Triturus cristatus*). Add a fourth sample to `SIMPLE_SAMPLES`:

- Query ID: `SAMPLE_D`
- Query definition: `12S_rRNA_newt`
- Query length: 180 bp (12S is shorter than 18S)
- One hit: accession `AJ005520`, title `Triturus cristatus 12S ribosomal RNA gene partial sequence`, identity 99.4%, alignment length 178, e-value `1e-85`, bit score 340.0

Re-run Sections 4 and 7. Does the diversity change? What is the new H' and J'?

### Exercise 3 — Interpret a result

You run the pipeline on a water sample and get back:

| query_id | resolved_species | identity_pct | confidence_flag |
|---|---|---|---|
| SEQ_01 | Oscillatoria sp. | 88.3% | low |
| SEQ_02 | unclassified | 79.1% | low |

Answer the following questions:
1. Can you report *Oscillatoria* at species level? Why or why not?
2. What does `unclassified` mean for SEQ_02 — is it definitely not a known organism, or something else?
3. What would you do next to improve these identifications? (Hint: think about database choice and re-running with a specialist database.)